# 1. Import Libraries

In [ ]:
import numpy as np
import tensorflow as tf
import itertools
import sys
import os
import gc
import json

# 2. Data & Model Paths

## 1. Folders

In [ ]:
scripts_folder = os.path.abspath(os.path.join('..', 'Scripts'))
data_folder = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
hyper_parameter_folder = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))

## 2. System paths

In [ ]:
sys.path.append(scripts_folder)

# 3. Import Data & Models

## 1. Import Models

In [ ]:
from model import Encoder, Decoder, VAE, RSVD

## 2. Import Data

In [ ]:
train_data_path = os.path.join(data_folder, 'train_data.npy')

try:
    train_data = np.load(train_data_path).astype(np.float32)
    num_items = train_data.shape[1]
    print(f"Data latih berhasil dimuat.")
    print(f"   Dimensi data: {train_data.shape} (Pengguna x Film)")
except FileNotFoundError:
    print("Error: File 'train_data.npy' tidak ditemukan. Pastikan Anda sudah menjalankan preprocessing.")

# 4. Hyperparameter Tuning

## 1.VAE

### 1. Hyperparameter Search Space

In [ ]:
vae_param_grid = {
    'latent_dim': [10, 20, 50, 100, 150, 200],         # Sangat padat hingga sangat detail
    'learning_rate': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],   # Rentang kecepatan belajar yang sangat luas
    'batch_size': [32, 64, 128, 256, 512],             # Ukuran batch
    'dropout_rate': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6],    # Mencari titik keseimbangan overfitting
    'hidden_dims': [
        [512, 256],               # Standar
        [256, 128],               # Dangkal
        [512, 256, 128],          # Dalam
        [1024, 512, 256]          # Sangat Dalam (Kapasitas memori besar)
    ]
}

# Membuat semua kemungkinan kombinasi
vae_keys, vae_values = zip(*vae_param_grid.items())
vae_combinations = [dict(zip(vae_keys, v)) for v in itertools.product(*vae_values)]

best_vae_loss = float('inf')
best_vae_params = None
best_vae_model = None

### 2. Grid Search

#### 1. Load Progress

In [ ]:
best_vae_path = os.path.join(hyper_parameter_folder, 'best_vae_weights.weights.h5')
vae_progress_file = os.path.join(hyper_parameter_folder, 'tuning_progress_vae.json')

if os.path.exists(vae_progress_file):
    with open(vae_progress_file, 'r') as f:
        progress_data = json.load(f)
    
    best_vae_loss = progress_data['best_loss']
    best_vae_params = progress_data['best_params']
    completed_indices = progress_data['completed_indices']
    
    print(f"\n[INFO] Melanjutkan sesi sebelumnya...")
    print(f"[INFO] {len(completed_indices)} kombinasi sudah dievaluasi.")
    print(f"[INFO] Loss terbaik saat ini: {best_vae_loss:.4f}")
else:
    best_vae_loss = float('inf')
    best_vae_params = None
    completed_indices = []

#### 2. Looping

In [ ]:
for i, params in enumerate(vae_combinations):
    # Lewati iterasi jika kombinasi ini sudah pernah dikerjakan sebelumnya
    if i in completed_indices:
        continue
    
    print(f"\n[{i+1}/{len(vae_combinations)}] Menguji VAE | Params: {params}")
    
    encoder = Encoder(hidden_dims=params['hidden_dims'], latent_dim=params['latent_dim'], dropout_rate=params['dropout_rate'])
    decoder = Decoder(hidden_dims=params['hidden_dims'][::-1], output_dim=num_items)
    vae = VAE(encoder, decoder)
    
    # Memasukkan 1 baris data agar Keras mengenali bentuk inputnya
    _ = vae(train_data[:1]) 
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=params['learning_rate'])
    vae.compile(optimizer=optimizer)
    
    # Melatih model
    history = vae.fit(train_data, train_data, epochs=15, batch_size=params['batch_size'], verbose=0)
    
    final_loss = history.history['loss'][-1]
    print(f"   -> Final Total Loss: {final_loss:.4f}")
    
    # Sistem Auto-Save jika loss lebih baik
    if final_loss < best_vae_loss:
        print(f"   🌟 [NEW BEST FOUND!] Loss turun ke {final_loss:.4f}. Menyimpan model...")
        best_vae_loss = final_loss
        best_vae_params = params
        best_vae_model = vae
        
        # Menyimpan bobot (sekarang tidak akan error lagi!)
        vae.save_weights(best_vae_path)

    completed_indices.append(i)
    with open(vae_progress_file, 'w') as f:
        json.dump({
            'best_loss': best_vae_loss,
            'best_params': best_vae_params,
            'completed_indices': completed_indices
        }, f)

    tf.keras.backend.clear_session() 
    del encoder, decoder, vae, history
    gc.collect()

print("\n==================================================")
print(f"[PENCARIAN SELESAI] Hasil Terbaik VAE:")
print(f"Parameter: {best_vae_params}")
print(f"Loss Terendah: {best_vae_loss:.4f}")
print(f"Bobot tersimpan di folder: {best_vae_path}")
print("==================================================")

### 3. Construct Best VAE Model

#### 1. Read Progress Files

In [ ]:
try:
    with open(vae_progress_file, 'r') as f:
        best_vae_params = json.load(f)['best_params']
    print(f"[INFO] Blueprint parameter terbaik ditemukan: {best_vae_params}")
except FileNotFoundError:
    print("[ERROR] File tuning_progress_vae.json tidak ditemukan.")

#### 2. Construct Model

In [ ]:
best_encoder = Encoder(
    hidden_dims=best_vae_params['hidden_dims'], 
    latent_dim=best_vae_params['latent_dim'], 
    dropout_rate=best_vae_params['dropout_rate']
)

best_decoder = Decoder(
    hidden_dims=best_vae_params['hidden_dims'][::-1], 
    output_dim=num_items
)

best_vae = VAE(best_encoder, best_decoder)

#### 3. Insert Weights

In [ ]:
_ = best_vae(train_data[:1])
best_vae.load_weights(best_vae_path)

### 4. Extract Latent Representation

In [ ]:
best_batch = best_vae_params['batch_size']
z_mean, _ = best_vae.encoder.predict(train_data, batch_size=best_batch)

latent_matrix_Z = z_mean
print(latent_matrix_Z.shape)

## 2. RSVD

### 1. Hyperparameter Search Space

In [ ]:
rsvd_param_grid = {
    'n_factors': [10, 20, 30, 50, 100, 150],              # Dimensi laten faktorisasi
    'learning_rate': [0.001, 0.005, 0.01, 0.05, 0.1],     # Kecepatan belajar SGD
    'lambda_reg': [0.001, 0.01, 0.05, 0.1, 0.2, 0.5]      # Variasi penalti untuk meredam overfitting
}

rsvd_keys, rsvd_values = zip(*rsvd_param_grid.items())
rsvd_combinations = [dict(zip(rsvd_keys, v)) for v in itertools.product(*rsvd_values)]

best_rsvd_error = float('inf')
best_rsvd_params = None
best_rsvd_model = None

### 2. Grid Search

#### 1. Load Progress

In [ ]:
rsvd_progress_file = os.path.join(hyper_parameter_folder, 'tuning_progress_rsvd.json')

if os.path.exists(rsvd_progress_file):
    with open(rsvd_progress_file, 'r') as f:
        progress_data = json.load(f)
    
    best_rsvd_error = progress_data['best_error']
    best_rsvd_params = progress_data['best_params']
    completed_indices = progress_data['completed_indices']
    
    print(f"\n[INFO] Melanjutkan sesi RSVD sebelumnya...")
    print(f"[INFO] {len(completed_indices)} kombinasi sudah dievaluasi.")
    print(f"[INFO] MSE terbaik saat ini: {best_rsvd_error:.4f}")
else:
    best_rsvd_error = float('inf')
    best_rsvd_params = None
    completed_indices = []

#### 2. Looping

In [ ]:
for i, params in enumerate(rsvd_combinations):
    # Lewati iterasi jika kombinasi ini sudah pernah dikerjakan
    if i in completed_indices:
        continue
    
    print(f"\n[{i+1}/{len(rsvd_combinations)}] Menguji RSVD | Params: {params}")
    
    # Inisialisasi model RSVD dengan parameter saat ini
    rsvd = RSVD(
        n_factors=params['n_factors'], 
        learning_rate=params['learning_rate'], 
        lambda_reg=params['lambda_reg'], 
        epochs=30  # Epoch kecil untuk tuning
    )
    
    # Latih RSVD menggunakan matriks laten Z
    rsvd.fit(latent_matrix_Z)
    
    # Evaluasi seberapa baik dekomposisi merekonstruksi Z (Menghitung MSE)
    # Z_rekonstruksi = U * Sigma * V^T
    reconstructed_Z = np.dot(np.dot(rsvd.U, rsvd.Sigma), rsvd.V.T)
    mse_error = np.mean(np.square(latent_matrix_Z - reconstructed_Z))
    
    print(f"   -> MSE Rekonstruksi Z: {mse_error:.4f}")
    
    # Cek apakah ini kombinasi terbaik
    if mse_error < best_rsvd_error:
        print(f"   🌟 [NEW BEST FOUND!] MSE turun ke {mse_error:.4f}. Menyimpan model...")
        best_rsvd_error = mse_error
        best_rsvd_params = params
        best_rsvd_model = rsvd

        # Menyimpan komponen RSVD jika mendapat skor terbaik
        np.save(os.path.join(hyper_parameter_folder, 'best_U.npy'), rsvd.U)
        np.save(os.path.join(hyper_parameter_folder, 'best_Sigma.npy'), rsvd.Sigma)
        np.save(os.path.join(hyper_parameter_folder, 'best_V.npy'), rsvd.V)

    completed_indices.append(i)
    with open(rsvd_progress_file, 'w') as f:
        json.dump({
            'best_error': best_rsvd_error,
            'best_params': best_rsvd_params,
            'completed_indices': completed_indices
        }, f)
        
    # Bersihkan variabel array raksasa dari memori untuk jaga-jaga
    del rsvd, reconstructed_Z
    gc.collect()

print("\n==================================================")
print(f"[HASIL TERBAIK RSVD]")
print(f"Parameter Terbaik: {best_rsvd_params}")
print(f"MSE Terendah pada Ruang Laten: {best_rsvd_error:.4f}")
print("==================================================")